# Study 883 — Mid-Cap Sweet Spot — the teardown

The excess-vs-excess Sharpe race, the paired-bootstrap advantage CIs, the HAC *t* on the cash-independent pairwise difference, the four-era myth-check, the costed dollar-neutral spread, and the planted-edge synthetic control.

In [1]:
R = {'as_of': '2026-06-30', 'fingerprint': 'd294f0cdb517', 'common_n': 4801, 'common': '2007-05 -> 2026-06', 'ijh_sh': 0.453, 'mdy_sh': 0.44, 'spy_sh': 0.542, 'iwm_sh': 0.394, 'ijh_ret': 11.58, 'spy_ret': 12.12, 'iwm_ret': 11.15, 'ijh_vol': 22.5, 'spy_vol': 19.8, 'iwm_vol': 24.8, 'ijh_dd': -55.1, 'spy_dd': -55.2, 'iwm_dd': -58.6, 'adv_spy': -0.089, 'adv_spy_lo': -0.225, 'adv_spy_hi': 0.05, 'adv_iwm': 0.06, 'adv_iwm_lo': -0.038, 'adv_iwm_hi': 0.169, 'ijh_spy_d': 1.74, 'ijh_spy_t': 1.19, 'ijh_iwm_d': 0.44, 'ijh_iwm_t': 0.38, 'mdy_spy_d': 0.95, 'mdy_spy_t': 0.67, 'mdy_spy_n': 7839, 'mdy_iwm_d': 0.32, 'mdy_iwm_t': 0.27, 'era1_d': 3.21, 'era1_t': 0.94, 'era2_d': 3.94, 'era2_t': 1.48, 'era3_d': 1.49, 'era3_t': 0.74, 'era4_d': -3.5, 'era4_t': -1.23, 'cost_spy_g': 1.74, 'cost_spy_n': 1.0, 'cost_spy_t': 0.68, 'cost_iwm_g': 0.44, 'cost_iwm_n': -0.3, 'cost_iwm_t': -0.25, 'charge': 0.74, 'null_beats': 1, 'planted_advL': 1.011, 'planted_tL': 5.78, 'planted_advS': 1.11, 'planted_tS': 5.02}

## The race — excess-of-cash Sharpe (2007-2026 common window)

Every leg minus BIL cash, so the race is apples-to-apples.

In [2]:
print(f"n = {R['common_n']} days ({R['common']})")
for tag, sh, rt, vl, dd in [('IJH mid ', R['ijh_sh'], R['ijh_ret'], R['ijh_vol'], R['ijh_dd']),
                            ('SPY large', R['spy_sh'], R['spy_ret'], R['spy_vol'], R['spy_dd']),
                            ('IWM small', R['iwm_sh'], R['iwm_ret'], R['iwm_vol'], R['iwm_dd'])]:
    print(f"  {tag}: exSharpe {sh:.3f}  ret {rt:+.2f}%  vol {vl:.1f}%  maxDD {dd:.1f}%")
print('mid sits BELOW large and just above small -> fails beats-both.')

n = 4801 days (2007-05 -> 2026-06)
  IJH mid : exSharpe 0.453  ret +11.58%  vol 22.5%  maxDD -55.1%
  SPY large: exSharpe 0.542  ret +12.12%  vol 19.8%  maxDD -55.2%
  IWM small: exSharpe 0.394  ret +11.15%  vol 24.8%  maxDD -58.6%
mid sits BELOW large and just above small -> fails beats-both.


## The advantage — mid excess-Sharpe minus each neighbour, paired block bootstrap

2,000 draws, 21-day blocks, resampled jointly to keep the cross-correlation.

In [3]:
print(f"IJH - SPY : adv {R['adv_spy']:+.3f}  95% CI [{R['adv_spy_lo']:+.3f}, {R['adv_spy_hi']:+.3f}]  -> spans 0")
print(f"IJH - IWM : adv {R['adv_iwm']:+.3f}  95% CI [{R['adv_iwm_lo']:+.3f}, {R['adv_iwm_hi']:+.3f}]  -> spans 0")

IJH - SPY : adv -0.089  95% CI [-0.225, +0.050]  -> spans 0
IJH - IWM : adv +0.060  95% CI [-0.038, +0.169]  -> spans 0


## The pairwise return difference — cash-independent, full tape

`mid − large` doesn't need the cash leg, so MDY reaches back to 1995.

In [4]:
print(f"IJH - SPY : {R['ijh_spy_d']:+.2f}%/yr  HAC t {R['ijh_spy_t']:+.2f}")
print(f"IJH - IWM : {R['ijh_iwm_d']:+.2f}%/yr  HAC t {R['ijh_iwm_t']:+.2f}")
print(f"MDY - SPY : {R['mdy_spy_d']:+.2f}%/yr  HAC t {R['mdy_spy_t']:+.2f}  (n={R['mdy_spy_n']}, since 1995)")
print(f"MDY - IWM : {R['mdy_iwm_d']:+.2f}%/yr  HAC t {R['mdy_iwm_t']:+.2f}")
print('sign-correct (mid out-returns both) but NONE clears |t|=2.')

IJH - SPY : +1.74%/yr  HAC t +1.19
IJH - IWM : +0.44%/yr  HAC t +0.38
MDY - SPY : +0.95%/yr  HAC t +0.67  (n=7839, since 1995)
MDY - IWM : +0.32%/yr  HAC t +0.27
sign-correct (mid out-returns both) but NONE clears |t|=2.


## Robustness — MDY − SPY by era (the myth-check)

In [5]:
for lbl, d, t in [('1995-2002', R['era1_d'], R['era1_t']), ('2003-2009', R['era2_d'], R['era2_t']),
                  ('2010-2016', R['era3_d'], R['era3_t']), ('2017-2026', R['era4_d'], R['era4_t'])]:
    flag = '  <- REVERSED' if d < 0 else ''
    print(f"  {lbl}: {d:+.2f}%/yr  HAC t {t:+.2f}{flag}")

  1995-2002: +3.21%/yr  HAC t +0.94
  2003-2009: +3.94%/yr  HAC t +1.48
  2010-2016: +1.49%/yr  HAC t +0.74
  2017-2026: -3.50%/yr  HAC t -1.23  <- REVERSED


## The costed spread — long mid / short neighbour (dollar-neutral)

50 bps/yr borrow on the short + 2 sides × 3 bps × 4 rebalances/yr.

In [6]:
print(f"long IJH / short SPY: gross {R['cost_spy_g']:+.2f} -> net {R['cost_spy_n']:+.2f}%/yr (t {R['cost_spy_t']:+.2f})")
print(f"long IJH / short IWM: gross {R['cost_iwm_g']:+.2f} -> net {R['cost_iwm_n']:+.2f}%/yr (t {R['cost_iwm_t']:+.2f})")
print(f"charge = {R['charge']:.2f}%/yr; only the reversed leg is net-positive, at t=+0.68.")

long IJH / short SPY: gross +1.74 -> net +1.00%/yr (t +0.68)
long IJH / short IWM: gross +0.44 -> net -0.30%/yr (t -0.25)
charge = 0.74%/yr; only the reversed leg is net-positive, at t=+0.68.


## Synthetic positive control — the machinery is unbiased

Live: a common market factor drives large/mid/small; `edge` lifts mid's excess mean. The detector must fire on a planted edge and stay quiet on the null.

In [7]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from midcap import data, strategy as st
beats = 0
for s in range(20):
    sig = st.synthetic_detect(data.synthetic_world(n_days=3000, edge=0.0, seed=883+s))
    beats += int(sig['beats_both'] and abs(sig['t_large'])>=2 and abs(sig['t_small'])>=2)
planted = st.synthetic_detect(data.synthetic_world(n_days=3000, edge=0.0006, seed=883))
print(f"null (edge=0), 20 seeds: strict-significant-beats-both in {beats}/20 (~nominal 5%)")
print(f"planted (edge=0.0006): adv vs large {planted['adv_large']:+.3f} (t {planted['t_large']:+.2f}), "
      f"vs small {planted['adv_small']:+.3f} (t {planted['t_small']:+.2f}), beats_both={planted['beats_both']}")

null (edge=0), 20 seeds: strict-significant-beats-both in 1/20 (~nominal 5%)
planted (edge=0.0006): adv vs large +1.011 (t +5.78), vs small +1.110 (t +5.02), beats_both=True


## Verdict

- **Signal — WEAK.** The 'mid beats BOTH' sweet-spot claim does not clear a robust bar. Mid's excess Sharpe (0.453) sits *below* large (0.542) and barely above small (0.394); both advantage CIs span zero. The long-run return tilt is sign-correct (MDY − SPY +0.95%/yr since 1995) but never significant (best HAC *t* = +1.19) and it **reversed** (-3.50%/yr in 2017-2026). A fragile, era-dependent tilt. The synthetic control fires cleanly on a planted edge (*t* ≈ 5-6) and fires on 1/20 nulls, so the detector is honest.
- **Tradability — MIRAGE.** No costed spread clears the bar: long-mid/short-large nets +1.00%/yr at *t* = +0.68 (and on the leg that inverted), long-mid/short-small nets -0.30%/yr. The Sharpe edge over small is a lower-return / similar-vol artifact — a Mirage after costs.